# Guesty → reservations sheet (one clean output)

Takes the two Guesty exports (check-out + check-in) plus your current Google Sheet
(downloaded as CSV) and writes **one file — your full, corrected sheet**: existing rows
kept as-is, superseded rows removed, and new/updated rows added.

## Each run
1. Drop the latest Guesty **check-out** and **check-in** CSVs (6-digit filenames) into `inputs/`.
2. Download your current Google Sheet as CSV into `inputs/` too.
3. **Kernel → Restart & Run All.**
4. Open **`sheet_updated.csv`**, select all, copy, and in your Google Sheet click **cell A1** and **paste**.

Pasting over A1 (rather than File → Import → Replace) keeps your **checkboxes working** —
paste replaces the values but leaves the checkbox formatting on the cells intact.

## What the output contains — rows matched on (Property, Date)
- **Existing rows** you already had: kept **verbatim** (manual edits + checkbox ticks preserved).
- **New** reservations: added.
- **Updated** (same property & date, but **guest / confirmation code / a time changed**):
  the new row is added and the **old row is dropped** — so it's already "deleted" for you.
  Updated rows **carry over the old row's checkbox ticks**.
- **Unchanged** rows (confirmation code, guest, both times all match): left as they were.

Times compare *normalized* (`4:00 PM` = `04:00 PM`); guest names ignore extra spaces/caps —
so only real changes replace a row.

## Notes
- Your sheet has ~2,000 pre-formatted empty rows below the data, so the pasted block lands
  within the checkbox-formatted range. If the data ever grows past that range, select the
  checkbox columns and extend the tick-box formatting down once.
- City is filled from `property_to_city.csv` + your sheet (exact → canonical → street level).
- A safety check stops the run if the dropped sheet isn't your reservations layout.
- **Fully hands-off option:** to have the code delete/update rows in the live sheet directly
  (no copy-paste at all), it would connect via the Google Sheets API — a one-time auth setup.
  Ask if you'd like that.

In [ ]:
import os, glob, re
from datetime import datetime
import pandas as pd
from processing import process_reservations, _canonical_key

# Drop the two Guesty exports + your downloaded sheet into this folder each run:
INPUT_DIR = "inputs"
# One clean output: the FULL corrected sheet (old superseded rows removed, new +
# updated rows added). Paste it over cell A1 of your sheet.
OUTPUT_CSV = "sheet_updated.csv"
# Optional: force a specific downloaded-sheet filename; None = newest in inputs/:
SHEET_CSV = None


def newest_with_columns(required, name_re=None):
    """Newest CSV in INPUT_DIR whose header has all `required` columns
    (and, if given, whose filename matches `name_re`)."""
    best, best_mtime = None, -1.0
    for path in glob.glob(os.path.join(INPUT_DIR, "*.csv")):
        if name_re and not name_re.match(os.path.basename(path)):
            continue
        try:
            cols = [c.strip() for c in pd.read_csv(path, nrows=0).columns]
        except Exception:
            continue
        if all(c in cols for c in required):
            m = os.path.getmtime(path)
            if m > best_mtime:
                best, best_mtime = path, m
    return best


os.makedirs(INPUT_DIR, exist_ok=True)
# Guesty exports are named with a leading 6-digit id; pick the newest of each type.
SIX_DIGIT = re.compile(r"^\d{6}")
CHECKOUT_CSV = newest_with_columns(["CHECK-OUT DATE"], SIX_DIGIT)
CHECKIN_CSV = newest_with_columns(["CHECK-IN DATE"], SIX_DIGIT)
if SHEET_CSV is None:
    SHEET_CSV = newest_with_columns(["Confirmation Code", "Property", "Date"])

print("Input folder  :", os.path.abspath(INPUT_DIR))
print("Check-out CSV :", CHECKOUT_CSV)
print("Check-in  CSV :", CHECKIN_CSV)
print("Current sheet :", SHEET_CSV or "(none found -> every row will be treated as new)")
assert CHECKOUT_CSV and CHECKIN_CSV, f"Put the Guesty check-out & check-in CSVs (6-digit names) in ./{INPUT_DIR}/"

In [ ]:
# 1) Transform the two Guesty exports -> native reservations columns
co = pd.read_csv(CHECKOUT_CSV); co.columns = co.columns.str.strip()
ci = pd.read_csv(CHECKIN_CSV);  ci.columns = ci.columns.str.strip()
candidates = process_reservations(co, ci)   # City filled in step 2b below

# 2) Load the current sheet -- provides what already exists, the column layout,
#    the existing rows (kept verbatim in the output), and city values.
if SHEET_CSV and os.path.exists(SHEET_CSV):
    sheet = pd.read_csv(SHEET_CSV, dtype=str).fillna("")
    sheet.columns = sheet.columns.str.strip()
else:
    sheet = pd.DataFrame(columns=list(candidates.columns))

# Safety guard: make sure the dropped sheet really is your reservations layout.
if len(sheet):
    signature = {"Confirmation Code", "Guest", "Property", "Date"}
    missing = signature - set(sheet.columns)
    if missing:
        raise ValueError(
            f"'{SHEET_CSV}' doesn't look like your reservations sheet "
            f"(missing columns: {sorted(missing)}).\n"
            f"Columns found: {list(sheet.columns)}\n"
            f"Drop the correct downloaded sheet into ./{INPUT_DIR}/ and remove any "
            f"other sheet-like CSVs from that folder."
        )

# 2b) Fill City from property_to_city.csv + the sheet. Match on exact name, then
#     canonical key, then street level (number + street, ignoring unit).
_unit_tok = re.compile(r"^(?:CH|[A-Za-z](?:-[A-Za-z])?|\d+|[0-9A-Za-z]*&[0-9A-Za-z&\-]*)$")
def _street_key(prop):
    toks = str(prop).strip().split()
    while len(toks) > 2 and _unit_tok.match(toks[-1]):
        toks.pop()
    return " ".join(toks).lower()

_city_exact, _city_canon, _city_street = {}, {}, {}
def _add_city(p, c):
    if str(c).strip():
        _city_exact.setdefault(str(p).strip(), c)
        _city_canon.setdefault(_canonical_key(p), c)
        _city_street.setdefault(_street_key(p), c)

if os.path.exists("property_to_city.csv"):
    _ref = pd.read_csv("property_to_city.csv", dtype=str).fillna("")
    for _, r in _ref.iterrows():
        _add_city(r["Property"], r["City"])
if len(sheet) and {"Property", "City"} <= set(sheet.columns):
    for _, r in sheet.iterrows():
        _add_city(r["Property"], r["City"])

def _resolve_city(p):
    return (_city_exact.get(str(p).strip())
            or _city_canon.get(_canonical_key(p))
            or _city_street.get(_street_key(p), ""))

candidates["City"] = [_resolve_city(p) for p in candidates["Property"]]

# 3) Match new rows' Date display format to the sheet (e.g. add ' 00:00:00' if used)
date_key = lambda v: str(v).strip()[:10]
if len(sheet) and sheet["Date"].astype(str).str.contains(r"\d\d:\d\d").any():
    candidates["Date"] = candidates["Date"].map(lambda v: f"{date_key(v)} 00:00:00")

# 4) Classify each candidate against the sheet on (Property, Date). A row is the
#    SAME reservation only if Confirmation Code, Guest, AND both (normalized) times
#    all match; if ANY differ it's an UPDATE (keep new + drop the old row).
norm = lambda c: "".join(ch for ch in str(c).lower() if ch.isalnum())

def _norm_time(t):
    t = str(t).strip()
    if not t:
        return ""
    for fmt in ("%I:%M %p", "%I:%M:%S %p", "%H:%M", "%H:%M:%S"):
        try:
            return datetime.strptime(t, fmt).strftime("%I:%M %p")
        except ValueError:
            pass
    return t.upper().replace(" ", "")

def _norm_guest(g):
    return " ".join(str(g).split()).casefold()

def _sheet_col(pipeline_name):
    tgt = norm(pipeline_name)
    return next((c for c in sheet.columns if norm(c) == tgt), None)

sheet_co_col = _sheet_col("Check-out Time")
sheet_ci_col = _sheet_col("Check-in Time")

def _sig(row, co_col, ci_col):
    return (
        str(row.get("Confirmation Code", "")).strip().upper(),
        _norm_guest(row.get("Guest", "")),
        _norm_time(row.get(co_col, "")),
        _norm_time(row.get(ci_col, "")),
    )

existing_by_key = {}
for i, r in sheet.iterrows():
    key = (str(r["Property"]).strip(), date_key(r["Date"]))
    existing_by_key.setdefault(key, []).append({
        "row": i + 2,                        # Google Sheet row number (header is row 1)
        "sig": _sig(r, sheet_co_col, sheet_ci_col),
        "data": r,
    })

keep_idx, carry_rows, delete_rows = [], [], []
n_new = n_updated = n_unchanged = 0
for j, c in candidates.iterrows():
    key = (str(c["Property"]).strip(), date_key(c["Date"]))
    matches = existing_by_key.get(key)
    if not matches:
        keep_idx.append(j); carry_rows.append(None); n_new += 1; continue
    csig = _sig(c, "Check-out Time", "Check-in Time")
    if any(m["sig"] == csig for m in matches):
        n_unchanged += 1; continue            # identical reservation already present
    keep_idx.append(j); carry_rows.append(matches[0]["data"]); n_updated += 1
    delete_rows.extend(matches)                # this slot's old row(s) are superseded

new_rows = candidates.loc[keep_idx].reset_index(drop=True)

# 5) Detect checkbox columns (values are TRUE/FALSE)
def _is_checkbox(col):
    v = sheet[col].str.strip()
    v = v[v != ""]
    return len(v) > 0 and v.str.upper().isin(["TRUE", "FALSE"]).mean() >= 0.8
checkbox_cols = [c for c in sheet.columns if _is_checkbox(c)] if len(sheet) else []

# 6) Build the new/updated rows, aligned to the sheet's exact columns. Checkbox
#    columns: UPDATED rows carry over the old row's tick; new rows get FALSE.
pipe_by_norm = {norm(c): c for c in new_rows.columns}
target_cols = list(sheet.columns) if len(sheet.columns) else list(candidates.columns)
to_append = pd.DataFrame("", index=range(len(new_rows)), columns=target_cols)
for col in target_cols:
    src = pipe_by_norm.get(norm(col))
    if src is not None:
        to_append[col] = new_rows[src].values
    elif col in checkbox_cols:
        vals = []
        for carry in carry_rows:
            old = str(carry[col]).strip() if (carry is not None and col in carry.index) else ""
            vals.append(old if old else "FALSE")
        to_append[col] = vals

# 7) ONE clean output: the full corrected sheet = existing DATA rows (minus the
#    ones superseded by an update) + the new/updated rows. Existing rows are kept
#    verbatim (manual edits + checkbox ticks preserved); superseded rows are left
#    out (the "deletion"); and the sheet's empty/checkbox-only filler rows are
#    dropped so the result is compact and pastes within the checkbox-formatted range.
delete_row_nums = {m["row"] for m in delete_rows}
if len(sheet):
    not_empty = ~((sheet["Date"].astype(str).str.strip() == "")
                  & (sheet["Property"].astype(str).str.strip() == ""))
    not_deleted = pd.Series([(i + 2) not in delete_row_nums for i in range(len(sheet))],
                            index=sheet.index)
    kept_existing = sheet[not_empty & not_deleted]
else:
    kept_existing = sheet
full = pd.concat([kept_existing, to_append], ignore_index=True) if len(kept_existing) else to_append.copy()
full = full.reindex(columns=target_cols).fillna("")
full.to_csv(OUTPUT_CSV, index=False)

missing_city = int((to_append.get("City", pd.Series(dtype=str)).astype(str).str.strip() == "").sum())
print(f"Existing sheet rows            : {len(sheet)}")
print(f"New                            : {n_new}")
print(f"Updated (guest/code/times)     : {n_updated}")
print(f"Unchanged                      : {n_unchanged}")
print(f"Old rows removed (superseded)  : {len(delete_rows)}")
print(f"-> {OUTPUT_CSV}: {len(full)} rows total  (paste over cell A1)")
print(f"New rows still missing City    : {missing_city}")

In [ ]:
# Preview: the new/updated rows added, and the old rows that were dropped
if len(to_append):
    print("=== New / updated rows added ===")
    print(to_append.head(20).to_string(index=False))
else:
    print("No new or updated rows -- the sheet is already up to date.")

if delete_rows:
    print(f"\n=== Old rows removed as superseded ({len(delete_rows)}) ===")
    print(pd.DataFrame([{
        "Sheet Row": m["row"], "Date": m["data"].get("Date", ""),
        "Property": m["data"].get("Property", ""), "Guest": m["data"].get("Guest", ""),
    } for m in delete_rows]).sort_values("Sheet Row").head(20).to_string(index=False))